# **Combining FPGA parallelism and machine learning for efficient image convolution**
Official implementation of the paper:

**Combining FPGA parallelism and machine learning for efficient image convolution**

Published in *Engineering Research Express*, 2026.

DOI: https://doi.org/10.1088/2631-8695/ae655e

Authors: Ameera Almomani, Doa’a Aloqoul and Abedalmuhdi Almomany*

## Description

This notebook reproduces the full-dataset experiments reported in the paper.

## Dataset

Place the dataset in the main workspace or update the dataset path if necessary.

## Citation

If you use this code, please cite the associated paper.

---


Copyright (c) 2026 Ameera Almomani

This software is licensed under the BSD 3-Clause License (https://opensource.org/license/bsd-3-clause)

 This source code is provided for academic and research purposes only.

Conditions of use:
1. You may use, modify, and distribute this code for *non-commercial*    research and educational purposes, provided that proper credit is given.
2. Commercial use of this code is *not permitted* without prior written consent.
3. Any publication or work that uses this code (in whole or in part) must cite the following paper: https://doi.org/10.1088/2631-8695/ae655e

---

In [ ]:
#=======================

import re
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_absolute_error, r2_score
import time
import numpy as np
from xgboost import XGBRegressor

# === Helper function to extract density & speed ===
def extract_density_speed(part_number):
    density_match = re.search(r'(\d+)[tTzZ]', part_number)
    density = int(density_match.group(1)) if density_match else None

    speed_match = re.search(r'-(\d+[A-Za-z]*)$', part_number)
    speed = f"-{speed_match.group(1)}" if speed_match else None

    return density, speed

# === Step 1: Load dataset ===
df = pd.read_csv("fpga_dataset2.csv")

# === Step 2: Extract features from PartNumber ===
df[["DENSITY", "SPEED"]] = df["FPGA_PART"].apply(
    lambda x: pd.Series(extract_density_speed(x))
)

# Drop PartNumber if not needed
df = df.drop(columns=["FPGA_PART"])

# === Step 3: Encode categorical columns ===
label_encoder_family = LabelEncoder()
df["FAMILY"] = label_encoder_family.fit_transform(df["FAMILY"])

label_encoder_speed = LabelEncoder()
df["SPEED"] = label_encoder_speed.fit_transform(df["SPEED"])

# === Step 4: Define features (X) and target (y) ===
X = df[[
    "WIDTH", "HEIGHT", "FAMILY", "DENSITY", "SPEED",
    "Frequency", "Latency", "DSP", "LUT", "FF", "BRAM"
]]
y = df["N"]

# === Step 5: Train/test split ===
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# === Standardize (important for SVR and KNN) ===
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# === Step 6: Initialize models ===
models = {
    "Decision Tree": DecisionTreeRegressor(random_state=42, max_depth=10),
    "Random Forest": RandomForestRegressor(random_state=42, n_estimators=100),
    "Support Vector Regression": SVR(kernel="rbf", C=100, gamma="scale"),
    "K-Nearest Neighbors": KNeighborsRegressor(n_neighbors=5),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42, n_estimators=200),
    "XGBoost": XGBRegressor(random_state=42, n_estimators=200, learning_rate=0.1, max_depth=6)
}

# === Step 7: Train & evaluate ===
results = {}
for name, model in models.items():
  if name in ["Support Vector Regression", "K-Nearest Neighbors"]:
    model.fit(X_train_scaled, y_train)

    start_time = time.perf_counter()
    y_pred = model.predict(X_test_scaled)
    end_time = time.perf_counter()

    total_time = end_time - start_time
    avg_time = total_time / len(X_test_scaled)
  else:
    model.fit(X_train, y_train)

    start_time = time.perf_counter()
    y_pred = model.predict(X_test)
    end_time = time.perf_counter()

    total_time = end_time - start_time
    avg_time = total_time / len(X_test)


  mae = mean_absolute_error(y_test, y_pred)
  r2 = r2_score(y_test, y_pred)

  results[name] = {
        "MAE": mae,
        "R2": r2,
        "Total_Time": total_time,
        "Avg_Time_per_sample": avg_time
        }
  print(f"\n{name}")
  print("  Mean Absolute Error:", mae)
  print("  R² Score:", r2)
  print(f"  Total Prediction Time: {total_time:.6f} sec")
  print(f"  Avg Time per Sample: {avg_time*1e6:.3f} µs")

# === Step 8: Visualization ===
'''mae_scores = [results[m]["MAE"] for m in results]
r2_scores = [results[m]["R2"] for m in results]

plt.figure(figsize=(10,4))
plt.subplot(1,2,1)
plt.bar(results.keys(), mae_scores, color="skyblue")
plt.ylabel("Mean Absolute Error")
plt.title("MAE Comparison")
plt.xticks(rotation=30)

plt.subplot(1,2,2)
plt.bar(results.keys(), r2_scores, color="lightgreen")
plt.ylabel("R² Score")
plt.title("R² Comparison")
plt.xticks(rotation=30)

plt.tight_layout()
plt.show()

'''
# === Step 8: Example prediction ===

sample = pd.DataFrame([{
    "WIDTH": 512,
    "HEIGHT": 512,
    "FAMILY": label_encoder_family.transform(["kintex7"])[0],
    "DENSITY": 70,
    "SPEED": label_encoder_speed.transform(["-1"])[0],
    "Frequency": 288,
    "Latency": 1.5,
    "DSP": 25,
    "LUT": 37,
    "FF": 9,
    "BRAM": 17
}])


print("\n=== Predictions for Single-Sample Case (Averaged) ===")

NUM_RUNS = 6000  #  increase to 5000 for more stability

for name, model in models.items():

    # Preprocess once (IMPORTANT: remove preprocessing from timing)
    if name in ["Support Vector Regression", "K-Nearest Neighbors"]:
        sample_input = scaler.transform(sample)
    else:
        sample_input = sample


    # Warm-up run (IMPORTANT: removes first-call overhead)
    model.predict(sample_input)

    # Timing loop
    start_time = time.perf_counter()
    for _ in range(NUM_RUNS):
        pred = model.predict(sample_input)
    end_time = time.perf_counter()

    avg_time_us = ((end_time - start_time) / NUM_RUNS)*1000

    print(f"{name}: {round(pred[0])} (raw: {pred[0]:.3f}) (Single-Sample_inf: {avg_time_us:.8f} ms)")



Decision Tree
  Mean Absolute Error: 0.07976973684210527
  R² Score: 0.9951510319422601
  Total Prediction Time: 0.001544 sec
  Avg Time per Sample: 1.269 µs

Random Forest
  Mean Absolute Error: 0.003240131578947369
  R² Score: 0.9999803760597605
  Total Prediction Time: 0.016128 sec
  Avg Time per Sample: 13.263 µs

Support Vector Regression
  Mean Absolute Error: 0.23081660846089655
  R² Score: 0.9964391327723242
  Total Prediction Time: 0.185104 sec
  Avg Time per Sample: 152.224 µs

K-Nearest Neighbors
  Mean Absolute Error: 0.4917763157894737
  R² Score: 0.9617491601955099
  Total Prediction Time: 0.011883 sec
  Avg Time per Sample: 9.772 µs

Gradient Boosting
  Mean Absolute Error: 0.1955168872571441
  R² Score: 0.9985507449462463
  Total Prediction Time: 0.008035 sec
  Avg Time per Sample: 6.607 µs

XGBoost
  Mean Absolute Error: 0.0021732195746153593
  R² Score: 0.9999993443489075
  Total Prediction Time: 0.017086 sec
  Avg Time per Sample: 14.051 µs

=== Predictions for Sing